# Actividad 5. Método de Uniformización

Se resuelven los ejercicios para programar de la actividad. Se considera la matriz de tasas $R$ del ejercicio 1 con $r = 6$.

In [1]:
import numpy as np
import math

# Matriz de tasas del ejercicio 1
R = np.array([[0, 2, 3, 0],
              [4, 0, 2, 0],
              [0, 2, 0, 2],
              [1, 0, 3, 0]], dtype=float)

N = len(R)

# Se calculan las tasas r_i (suma de cada renglon)
r_i = R.sum(axis=1)
print("Tasas r_i:", r_i)

# Se toma r como el maximo de las tasas, en este caso r = 6
r = max(r_i)
print("r =", r)

Tasas r_i: [5. 6. 4. 4.]
r = 6.0


In [2]:
# Se construye la matriz P gorro de acuerdo a la definicion:
# p_ij = 1 - r_i/r  si i = j
# p_ij = r_ij / r   si i != j
P_gorro = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i == j:
            P_gorro[i][j] = 1 - r_i[i]/r
        else:
            P_gorro[i][j] = R[i][j]/r

print("Matriz P gorro:")
print(P_gorro)

# Verificamos que sea estocastica (los renglones suman 1)
print("Suma de renglones:", P_gorro.sum(axis=1))

Matriz P gorro:
[[0.16666667 0.33333333 0.5        0.        ]
 [0.66666667 0.         0.33333333 0.        ]
 [0.         0.33333333 0.33333333 0.33333333]
 [0.16666667 0.         0.5        0.33333333]]
Suma de renglones: [1. 1. 1. 1.]


## Ejercicio 3

Se aproxima $P(t)$ con los primeros $M$ términos de la serie

$$P(t) = \sum_{k=0}^{\infty} e^{-rt}\frac{(rt)^k}{k!}\hat{P}^k$$

usando $M \approx \max\{rt + 5\sqrt{rt},\; 20\}$. Se calcula $P(0.5)$, $P(1)$ y $P(5)$.

In [3]:
def aproximar_P(t):
    # Se elige M de acuerdo a la propuesta
    M = math.ceil(max(r*t + 5*math.sqrt(r*t), 20))
    suma = np.zeros((N, N))
    A = np.identity(N)       # A guarda las potencias de P gorro, empieza en P^0 = I
    c = math.exp(-r*t)       # c guarda el coeficiente e^{-rt} (rt)^k / k!
    for k in range(M + 1):
        suma = suma + c*A
        A = A @ P_gorro          # siguiente potencia de P gorro
        c = c*(r*t)/(k + 1)      # siguiente coeficiente de Poisson
    return suma, M

In [4]:
# Inciso 1: se calcula P(0.5), P(1) y P(5)
P_05, M_05 = aproximar_P(0.5)
P_1, M_1 = aproximar_P(1)
P_5, M_5 = aproximar_P(5)

print("P(0.5) con M =", M_05)
print(np.round(P_05, 5))
print()
print("P(1) con M =", M_1)
print(np.round(P_1, 5))
print()
print("P(5) con M =", M_5)
print(np.round(P_5, 5))

P(0.5) con M = 20
[[0.25061 0.21696 0.38666 0.14577]
 [0.25313 0.23836 0.37441 0.13409]
 [0.16912 0.19361 0.4203  0.21696]
 [0.15802 0.15744 0.39833 0.28621]]

P(1) con M = 20
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]

P(5) con M = 58
[[0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]]


In [5]:
# Inciso 2: se verifica la ecuacion de Chapman-Kolmogorov P(1) = P(0.5)P(0.5)
producto = P_05 @ P_05

print("P(0.5)P(0.5):")
print(np.round(producto, 5))
print()
print("P(1):")
print(np.round(P_1, 5))
print()
# Diferencia maxima entre ambas matrices
diferencia = np.max(np.abs(P_1 - producto))
print("Diferencia maxima:", diferencia)

P(0.5)P(0.5):
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]

P(1):
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]

Diferencia maxima: 5.820333638384412e-07


Las dos matrices coinciden hasta el orden de $10^{-7}$, que es el error de truncar la serie, por lo que sí se verifica la ecuación de Chapman-Kolmogorov.

## Ejercicio 4

Se implementa el algoritmo de uniformización con tolerancia $\epsilon$. Ahora $M$ no se fija de antemano: se suman términos hasta que

$$\sum_{k=0}^{M} e^{-rt}\frac{(rt)^k}{k!} \geq 1 - \epsilon$$

lo que garantiza que la cola de la serie es menor que $\epsilon$.

In [6]:
def uniformizacion(t, eps):
    # Paso 4 del algoritmo
    A = P_gorro.copy()
    B = math.exp(-r*t)*np.identity(N)
    c = math.exp(-r*t)
    suma = c
    k = 1
    # Paso 5: se itera mientras la suma de coeficientes no alcance 1 - eps
    while suma < 1 - eps:
        c = c*(r*t)/k
        B = B + c*A
        A = A @ P_gorro
        suma = suma + c
        k = k + 1
    M = k - 1   # numero de terminos usados en la serie
    return B, M

In [7]:
# Se repite el ejercicio 3 con tolerancia eps = 0.00001
eps = 0.00001

B_05, M2_05 = uniformizacion(0.5, eps)
B_1, M2_1 = uniformizacion(1, eps)
B_5, M2_5 = uniformizacion(5, eps)

print("P(0.5) con el algoritmo, M =", M2_05)
print(np.round(B_05, 5))
print()
print("P(1) con el algoritmo, M =", M2_1)
print(np.round(B_1, 5))
print()
print("P(5) con el algoritmo, M =", M2_5)
print(np.round(B_5, 5))

P(0.5) con el algoritmo, M = 13
[[0.25061 0.21696 0.38666 0.14577]
 [0.25313 0.23836 0.37441 0.13409]
 [0.16912 0.19361 0.4203  0.21696]
 [0.15802 0.15744 0.39833 0.28621]]

P(1) con el algoritmo, M = 19
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]

P(5) con el algoritmo, M = 56
[[0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]]


In [8]:
# Comparacion con los resultados del ejercicio 3
print("Diferencia maxima en t = 0.5:", np.max(np.abs(P_05 - B_05)))
print("Diferencia maxima en t = 1:  ", np.max(np.abs(P_1 - B_1)))
print("Diferencia maxima en t = 5:  ", np.max(np.abs(P_5 - B_5)))

Diferencia maxima en t = 0.5: 1.3607613894017767e-06
Diferencia maxima en t = 1:   1.490024779948751e-06
Diferencia maxima en t = 5:   2.200128962071002e-06


**Comparación.** Con la tolerancia $\epsilon = 0.00001$ el algoritmo usó $M = 13$ para $t=0.5$, $M = 19$ para $t=1$ y $M = 56$ para $t=5$, mientras que la propuesta del ejercicio 3 usaba $M = 20$, $20$ y $58$ respectivamente. Los resultados coinciden hasta el orden de $10^{-6}$, es decir, dentro de la tolerancia pedida. La ventaja del algoritmo es que controla el error directamente y en general necesita menos términos.